In [1]:
import pandas as pd
import numpy as np

In [4]:
import os

os.listdir()

['01_read_and_join.ipynb',
 '02_create_labels.ipynb',
 '03_train_val_test_split.ipynb',
 '04_eda.ipynb',
 '05_feature_engineering.ipynb',
 'features.pkl',
 'image.png',
 'labeled_orders.csv',
 'rf_model.pkl']

In [5]:
orders = pd.read_csv("labeled_orders.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0


In [6]:
import pandas as pd
import os
import joblib

orders = pd.read_csv("labeled_orders.csv")

print("Shape:", orders.shape)
orders.head()

Shape: (96478, 9)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0


In [7]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [8]:
# Purchase time features
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour
orders["purchase_day_of_week"] = orders["order_purchase_timestamp"].dt.dayofweek
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

# Time between purchase and approval
orders["approval_delay_hours"] = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

# Estimated delivery duration
orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,purchase_hour,purchase_day_of_week,purchase_month,purchase_year,approval_delay_hours,estimated_delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,10,0,10,2017,0.178333,15.544063
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,20,1,7,2018,30.713889,19.137766
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,8,2,8,2018,0.276111,26.639711
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,19,5,11,2017,0.298056,26.188819
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,21,1,2,2018,1.030556,12.112049


In [9]:
target = "is_late"

drop_cols = [
    "is_late",
    "order_id",
    "customer_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "delivery_days",
    "delivery_delay_days",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
]

X = orders.drop(columns=drop_cols, errors="ignore")
y = orders[target]

print("Features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Features:
['purchase_hour', 'purchase_day_of_week', 'purchase_month', 'purchase_year', 'approval_delay_hours', 'estimated_delivery_days']

Target distribution:
is_late
0    88652
1     7826
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (67534, 6)
Validation: (14472, 6)
Test: (14472, 6)


In [11]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Missing values after imputation:")
print(np.isnan(X_train_imputed).sum())

Missing values after imputation:
0


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [13]:
feature_list = X.columns.tolist()

X_train_final = pd.DataFrame(X_train_scaled, columns=feature_list, index=X_train.index)

X_val_final = pd.DataFrame(X_val_scaled, columns=feature_list, index=X_val.index)

X_test_final = pd.DataFrame(X_test_scaled, columns=feature_list, index=X_test.index)

X_train_final.head()

,purchase_hour,purchase_day_of_week,purchase_month,purchase_year,approval_delay_hours,estimated_delivery_days
91117,-0.711472,1.139302,0.607893,-1.077843,-0.487437,-0.140655
75327,-0.523600,-0.895598,-0.011426,0.908270,0.977537,-0.143307
16436,0.415758,0.630577,0.607893,0.908270,-0.487261,-0.853272
57792,-0.523600,-1.404323,0.607893,0.908270,-0.488129,0.541015
72774,-0.335729,-0.895598,0.607893,0.908270,-0.490081,-2.091427


In [14]:
os.makedirs("artifacts", exist_ok=True)

In [ ]:
joblib.dump(imputer, "artifacts/imputer.pkl")
joblib.dump(scaler, "artifacts/scaler.pkl")
joblib.dump(feature_list, "artifacts/feature_list.pkl")

print("Transformers saved successfully.")

Transformers saved successfully.


In [16]:
X_train_final.to_csv("artifacts/X_train_features.csv", index=False)

X_val_final.to_csv("artifacts/X_val_features.csv", index=False)

X_test_final.to_csv("artifacts/X_test_features.csv", index=False)

y_train.to_csv("artifacts/y_train.csv", index=False)

y_val.to_csv("artifacts/y_val.csv", index=False)

y_test.to_csv("artifacts/y_test.csv", index=False)

print("Feature tables saved successfully.")

Feature tables saved successfully.


In [17]:
print("===== FINAL CHECK =====")

print("\nTrain shape:", X_train_final.shape)
print("Validation shape:", X_val_final.shape)
print("Test shape:", X_test_final.shape)

print("\nFeature list:")
for feature in feature_list:
    print("-", feature)

print("\nMissing values:")
print("Train:", X_train_final.isna().sum().sum())
print("Validation:", X_val_final.isna().sum().sum())
print("Test:", X_test_final.isna().sum().sum())

print("\nArtifacts:")
print(os.listdir("artifacts"))

===== FINAL CHECK =====

Train shape: (67534, 6)
Validation shape: (14472, 6)
Test shape: (14472, 6)

Feature list:
- purchase_hour
- purchase_day_of_week
- purchase_month
- purchase_year
- approval_delay_hours
- estimated_delivery_days

Missing values:
Train: 0
Validation: 0
Test: 0

Artifacts:
['feature_list.pkl', 'imputer.pkl', 'scaler.pkl', 'X_test_features.csv', 'X_train_features.csv', 'X_val_features.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']


In [2]:
import joblib

feature_list = joblib.load("artifacts/feature_list.pkl")

print(feature_list)

['purchase_hour', 'purchase_day_of_week', 'purchase_month', 'purchase_year', 'approval_delay_hours', 'estimated_delivery_days']
